# 04 Participant analysis and interpretation

This notebook demonstrates how recording-level counts become independent-participant rates and how inferential results are kept separate from descriptive patterns.

## 1. Run the complete synthetic pipeline

The public fixture contains only two participants, so it validates aggregation but deliberately does not run a three-group statistical comparison.

In [ ]:
from pathlib import Path
import csv, json, sys, tempfile
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from bangor_miami.pipeline import run_pipeline
with tempfile.TemporaryDirectory() as output:
    output = Path(output)
    summary = run_pipeline(ROOT / 'data/sample/chats', ROOT / 'data/sample/tsvs', ROOT / 'data/sample/sample_metadata.csv', output)
    participant_rows = list(csv.DictReader((output / 'participant_rates.csv').open()))
summary, participant_rows

## 2. Inspect the statistical API

The example below uses artificial rates solely to make every statistical output executable without redistributing participant data.

In [ ]:
from bangor_miami.statistics import inferential_analysis
synthetic_participants = []
for level, group in ((1, 'Low'), (3, 'Middle'), (5, 'High')):
    for index, rate in enumerate((2.0, 4.0, 6.0, 8.0)):
        synthetic_participants.append({'questionnaire_id': f'{group}{index}', 'education_level': level, 'education_group': group, 'switch_rate_per_1000_valid_tokens': rate + level})
synthetic_inference = inferential_analysis(synthetic_participants)
synthetic_inference['tests']

## 3. Verified private-corpus result

The same functions were run locally on 73 strictly matched participants. Welch ANOVA gave `F(2, 11.31) = 0.109, p = .8976`; Kruskal-Wallis gave `H(2) = 0.341, p = .8434`; and the ordinal education-level HC3 slope was `0.639, p = .6309`. None supports a reliable education effect.

The slight increase in group means is descriptive only. The small Low group (`n = 5`), conservative switch definition and unresolved mappings limit generalisation.